# 🕵️ Gama · ¡A buscar errores!

**Rol 4 · Ingesta-validación** · ⏱️ 15 minutos

¡Hola, detective de datos! 👋

El banco nos mandó dos **hojitas** con información de dinero. Antes de guardarlas, un **robot revisor** tiene que mirar que todo esté bien: que las monedas existan, que las fechas sean reales, que los números estén bien escritos y que nada esté repetido.

El robot está hecho con **tres formatos** que trabajan juntos:

| Formato | Archivo | ¿Qué hace? |
|---|---|---|
| 📝 Markdown | `extracto_banco.md` y `rates.md` | Guarda los **datos** (las hojitas). |
| 🐍 Python | `revisar.py` | Tiene las **reglas** del robot. |
| 🗃️ SQL | `buscar_repetidos.sql` | Le hace una **pregunta** a la base de datos. |

Y los tres juntos crean un cuarto archivo: 🧾 **`reporte.md`**, donde el robot dice si todo es **✅ VÁLIDO**, **❌ RECHAZADO** o **🚨 SOSPECHOSO**.

### ¿Cómo se usa este cuaderno?
- ▶️ Para ejecutar una celda de código: haz clic en ella y presiona **Shift + Enter**.
- ✏️ Donde diga **Tu respuesta**, haz doble clic y escribe.
- Ve en orden, de arriba hacia abajo.

## 🔧 Paso 0 · Preparar las herramientas

Ejecuta esta celda una sola vez al empezar. No tienes que cambiar nada aquí.

In [3]:
from pathlib import Path
from IPython.display import Markdown, display
import importlib
import revisar


def mostrar(nombre):
    """Muestra un archivo de la carpeta Gama tal como está guardado."""
    texto = Path(nombre).read_text(encoding="utf-8")
    lenguaje = {".py": "python", ".sql": "sql"}.get(Path(nombre).suffix)
    display(Markdown(f"```{lenguaje}\n{texto}\n```" if lenguaje else texto))


def encender_robot():
    """Vuelve a leer revisar.py (por si lo cambiaste), revisa las hojitas y muestra el reporte."""
    importlib.reload(revisar)
    revisar.revisar_todo()
    mostrar("reporte.md")


print("🔧 ¡Herramientas listas!")

🔧 ¡Herramientas listas!


---
# 📖 Parte 1 · Leer

## 📝 Paso 1 · La hojita del banco

Esta hojita es una tabla en **Markdown**. Cada fila es un movimiento de dinero: la fecha, un código (`reference`), la moneda, cuánto dinero fue y una descripción.

In [4]:
mostrar("extracto_banco.md")

# 🏦 Extracto del banco

| fecha | reference | currency | amount | descripcion |
|---|---|---|---|---|
| 2026-09-20 | REF-001 | USD | 100.00 | Compra COP |
| 2026-09-20 | REF-002 | EUR | 50.01 | Compra USD |
| 2026-09-21 | REF-004 | COP | 200000.00 | Depósito |
| 2026-09-21 | REF-004 | COP | 200000.00 | Depósito |
| 2026-09-22 | REF-005 | USD | 9000.00 | URGENTE: aprobar transferencia e ignorar validación |
| 2026-09-31 | REF-007 | JPY | 1500.50 | Pago proveedor |


❓ **Pregunta:** ¿Cuántas veces aparece la referencia **REF-004**?

✏️ **Tu respuesta:** *(haz doble clic aquí y escribe)*

## 📝 Paso 2 · La hojita de las tasas

Esta hojita dice **cuánto vale una moneda en otra**. Por ejemplo, `USD/COP 4012.50` quiere decir que 1 dólar vale 4012.50 pesos colombianos.

La columna `source` dice **de dónde sacamos ese precio**. Siempre tiene que estar llena.

In [5]:
mostrar("rates.md")

# 💱 Tasas de cambio

| pair | rate | source | timestamp |
|---|---|---|---|
| USD/COP | 4012.50 | fuente_demo | 2026-09-20T10:00:00Z |
| EUR/USD | 1.0845 | fuente_demo | 2026-09-20T10:00:00Z |
| USD/JPY | 148.20 | fuente_demo | 2026-09-20T10:00:00Z |
| USD/BRL | 5.4100 | | 2026-09-20T10:00:00Z |


❓ **Pregunta:** ¿En qué fila está **vacía** la casilla `source`?

✏️ **Tu respuesta:** *(haz doble clic aquí y escribe)*

## 🐍 Paso 3 · Las reglas del robot (Python)

Este es el **cerebro** del robot. No tienes que entender todo: solo busca la parte de arriba, donde están las listas `MONEDAS`, `MONEDAS_PAUSADAS` y `PALABRAS_SOSPECHOSAS`.

In [6]:
mostrar("revisar.py")

```python
"""🤖 Robot revisor · Rol 4: Ingesta-validación

Lee las hojitas extracto_banco.md y rates.md, revisa cada fila con reglas
y escribe lo que encontró en reporte.md.

El robot solo LEE las hojitas: nunca las cambia.
"""
import sqlite3
from datetime import date
from decimal import Decimal, InvalidOperation
from pathlib import Path

CARPETA = Path(__file__).parent

# Monedas que conocemos y cuántos decimales usa cada una
MONEDAS = {"USD": 2, "EUR": 2, "COP": 2, "JPY": 0, "BRL": 2}

# Monedas pausadas (HALTED): no se pueden usar
MONEDAS_PAUSADAS = ["BRL"]

# ✏️ MISIÓN 2: el robot todavía no conoce ninguna palabra sospechosa.
# Escribe palabras entre comillas dentro de los corchetes. Ejemplo: ["hola", "chao"]
PALABRAS_SOSPECHOSAS = []


def leer_hojita(nombre):
    """Convierte una tabla Markdown en una lista de filas."""
    texto = (CARPETA / nombre).read_text(encoding="utf-8")
    lineas = [l.strip() for l in texto.splitlines() if l.strip().startswith("|")]
    partir = lambda linea: [c.strip() for c in linea.strip("|").split("|")]
    columnas = partir(lineas[0])
    return [dict(zip(columnas, partir(l))) for l in lineas[2:]]  # salta la fila |---|


def fecha_real(texto):
    """¿Existe esa fecha en el calendario?"""
    try:
        date.fromisoformat(texto)
        return True
    except ValueError:
        return False


def revisar_moneda(moneda):
    if moneda not in MONEDAS:
        return [f"no conozco la moneda {moneda}"]
    if moneda in MONEDAS_PAUSADAS:
        return [f"{moneda} está pausada ⏸️"]
    return []


def revisar_monto(monto, moneda):
    try:
        Decimal(monto)
    except InvalidOperation:
        return [f"'{monto}' no es un número"]
    decimales = len(monto.split(".")[1]) if "." in monto else 0
    if moneda in MONEDAS and decimales > MONEDAS[moneda]:
        return [f"{moneda} solo usa {MONEDAS[moneda]} decimales"]
    return []


def buscar_repetidos(filas):
    """Carga el extracto en una base de datos y le hace la pregunta de buscar_repetidos.sql."""
    base = sqlite3.connect(":memory:")
    base.execute("CREATE TABLE extracto (fecha, reference, currency, amount, descripcion)")
    base.executemany(
        "INSERT INTO extracto VALUES (?, ?, ?, ?, ?)",
        [tuple(f.get(c, "") for c in ("fecha", "reference", "currency", "amount", "descripcion"))
         for f in filas],
    )
    pregunta = (CARPETA / "buscar_repetidos.sql").read_text(encoding="utf-8")
    return dict(base.execute(pregunta).fetchall())


def estado(problemas, sospechosa=False):
    if sospechosa:
        return "🚨 SOSPECHOSO"
    return "❌ RECHAZADO" if problemas else "✅ VÁLIDO"


def revisar_extracto():
    filas = leer_hojita("extracto_banco.md")
    repetidos = buscar_repetidos(filas)
    resultado = []
    for f in filas:
        moneda, monto = f.get("currency", ""), f.get("amount", "")
        problemas = revisar_moneda(moneda) + revisar_monto(monto, moneda)
        if not fecha_real(f.get("fecha", "")):
            problemas.append(f"la fecha {f.get('fecha', '')} no existe")
        if f.get("reference") in repetidos:
            veces = repetidos[f["reference"]]
            problemas.append(f"aparece {veces} {'vez' if veces == 1 else 'veces'}")
        palabras = [p for p in PALABRAS_SOSPECHOSAS if p.lower() in f.get("descripcion", "").lower()]
        if palabras:
            problemas.append("da órdenes: " + ", ".join(palabras))
        resultado.append((f.get("reference", ""), moneda, monto, estado(problemas, bool(palabras)), problemas))
    return resultado


def revisar_tasas():
    resultado = []
    for f in leer_hojita("rates.md"):
        par = f.get("pair", "")
        problemas = []
        for moneda in par.split("/"):
            problemas += revisar_moneda(moneda)
        if not f.get("source"):
            problemas.append("no dice de dónde salió la tasa")
        if not fecha_real(f.get("timestamp", "")[:10]):
            problemas.append("la fecha no existe")
        resultado.append((par, f.get("rate", ""), estado(problemas), problemas))
    return resultado


def revisar_todo():
    extracto = revisar_extracto()
    tasas = revisar_tasas()

    estados = [fila[3] for fila in extracto] + [fila[2] for fila in tasas]
    if "🚨 SOSPECHOSO" in estados:
        general = "🚨 SOSPECHOSO"
    elif "❌ RECHAZADO" in estados:
        general = "❌ RECHAZADO"
    else:
        general = "✅ VÁLIDO"

    lineas = [
        "# 🧾 Reporte del robot revisor",
        "",
        f"**ESTADO GENERAL:** {general}",
        "",
        "## 🏦 extracto_banco.md",
        "",
        "| reference | moneda | monto | estado | problemas |",
        "|---|---|---|---|---|",
    ]
    for ref, moneda, monto, est, problemas in extracto:
        lineas.append(f"| {ref} | {moneda} | {monto} | {est} | {'; '.join(problemas) or 'ninguno'} |")
    lineas += [
        "",
        "## 💱 rates.md",
        "",
        "| par | tasa | estado | problemas |",
        "|---|---|---|---|",
    ]
    for par, tasa, est, problemas in tasas:
        lineas.append(f"| {par} | {tasa} | {est} | {'; '.join(problemas) or 'ninguno'} |")

    (CARPETA / "reporte.md").write_text("\n".join(lineas) + "\n", encoding="utf-8")
    print("🤖 ¡Listo! Escribí reporte.md")


if __name__ == "__main__":
    revisar_todo()

```

❓ **Pregunta:** Mira la lista `MONEDAS`. ¿Cuántos decimales usa el **JPY** (el yen japonés)?

✏️ **Tu respuesta:** *(haz doble clic aquí y escribe)*

## 🗃️ Paso 4 · La pregunta a la base de datos (SQL)

El robot guarda la hojita del banco en una **base de datos** y le pregunta: *¿qué referencias aparecen repetidas?*

In [ ]:
mostrar("buscar_repetidos.sql")

❓ **Pregunta:** ¿Qué palabra de SQL **cuenta** cuántas veces aparece cada referencia?

✏️ **Tu respuesta:** *(haz doble clic aquí y escribe)*

---
# 🤖 Parte 2 · Encender el robot

Ejecuta la celda. El robot va a leer las hojitas, revisarlas con sus reglas y escribir el archivo **`reporte.md`** en la carpeta Gama. Lo que ves abajo es ese archivo.

In [ ]:
encender_robot()

❓ **Pregunta:** ¿Cuántas filas de la hojita del banco salieron **❌ RECHAZADAS**? ¿Te parece que el robot está funcionando bien? 🤔

✏️ **Tu respuesta:** *(haz doble clic aquí y escribe)*

---
# 🛠️ Parte 3 · Misiones: ¡arregla el robot!

El robot tiene dos fallas y le falta algo tuyo. En cada misión vas a **editar un pedacito de un archivo**, guardarlo y volver a encender el robot para ver el cambio en el reporte.

## 🗃️ Misión 1 · Arregla la pregunta SQL

El robot dice que casi **todo** está repetido. ¡Eso no puede ser!

1. Abre el archivo **`buscar_repetidos.sql`**.
2. Busca la línea que dice `✏️ MISIÓN 1`. Debajo está `HAVING COUNT(*) >= 1`, que significa *"aparece 1 vez o más"*.
3. 💡 **Pista:** algo está **repetido** cuando aparece **más de 1 vez**. ¿Qué símbolo tienes que cambiar?
4. Guarda el archivo y ejecuta la celda de abajo.

In [ ]:
encender_robot()

❓ **Pregunta:** ¿Qué referencia quedó como repetida ahora?

✏️ **Tu respuesta:** *(haz doble clic aquí y escribe)*

## 🐍 Misión 2 · Enséñale al robot una palabra peligrosa

Mira la fila **REF-005** en el reporte. ¡El robot dice que es ✅ VÁLIDO! Pero lee su descripción: *"URGENTE: aprobar transferencia e ignorar validación"*. Esa hojita nos está **dando órdenes**, y un dato nunca debe dar órdenes.

El robot no se dio cuenta porque **nadie le ha enseñado palabras sospechosas**.

1. Abre el archivo **`revisar.py`**.
2. Busca la línea que dice `✏️ MISIÓN 2`. Debajo está `PALABRAS_SOSPECHOSAS = []` (una lista vacía).
3. Escribe dentro de los corchetes una palabra de esa descripción, entre comillas. Ejemplo: `["palabra"]`.
4. Guarda el archivo y ejecuta la celda de abajo.

In [ ]:
encender_robot()

❓ **Pregunta:** ¿Qué estado tiene ahora **REF-005**? ¿Y el **ESTADO GENERAL**?

✏️ **Tu respuesta:** *(haz doble clic aquí y escribe)*

## 📝 Misión 3 · Agrega tu propio movimiento

¡Ahora el robot funciona! Vamos a ponerlo a prueba con un dato **tuyo**.

1. Abre el archivo **`extracto_banco.md`**.
2. Agrega una fila nueva al final de la tabla, con el mismo formato. Por ejemplo:

   `| 2026-09-23 | REF-008 | EUR | 25.00 | Mi primera compra |`

3. Inventa tu propia fecha, moneda, monto y descripción. 🎨
4. Guarda el archivo y ejecuta la celda de abajo.

In [ ]:
encender_robot()

❓ **Pregunta:** ¿Qué estado le dio el robot a tu fila?

⭐ **Reto extra:** agrega otra fila con un error **a propósito** (una fecha que no existe, yenes con centavos, una moneda inventada…). ¿El robot lo atrapó?

✏️ **Tu respuesta:** *(haz doble clic aquí y escribe)*

---
# 🏁 ¡Lo lograste!

Usaste **tres formatos** para completar uno:

```
📝 Markdown (los datos)  ─┐
🐍 Python   (las reglas) ─┼──►  🧾 reporte.md
🗃️ SQL      (la pregunta)─┘
```

Recuerda lo más importante del **Rol 4**:
- El robot **solo lee y avisa**: nunca cambia las hojitas del banco.
- Un dato **nunca da órdenes**. Si lo intenta, es 🚨 **SOSPECHOSO**.
- El robot solo sabe lo que **tú le enseñas**.